# CV Bot (NLP Project)

## Importing Libraraies

In [1]:
%pip install -q "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" \
    "langchain-google-genai>=2.0" "langchain-community>=0.3,<0.4" \
    "langchain-text-splitters>=0.3,<0.4" "faiss-cpu>=1.8" \
    "pypdf>=4.0" "python-dotenv>=1.0"
print("Done. Restart the kernel if this was the first install.")

Note: you may need to restart the kernel to use updated packages.
Done. Restart the kernel if this was the first install.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.6 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.12.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Loading API Key

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads the .env file in this folder
assert os.environ.get("GOOGLE_API_KEY"), \
    "No API key found. Create a .env file with GOOGLE_API_KEY=AIza..."
print("API key loaded:", os.environ["GOOGLE_API_KEY"][:7] + "...")

API key loaded: AQ.Ab8R...


# Setting up Language Model

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)
print("LLM ready:", llm.model)

LLM ready: models/gemini-2.5-flash


# Loading the documents

In [4]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader
CV_PATH = Path("CV.txt")
documents = TextLoader(str(CV_PATH), encoding="utf-8").load()
print(f"Loaded {len(documents)} document(s) from {CV_PATH}")
for d in documents:
    print(" -", Path(d.metadata["source"]).name, f"({len(d.page_content)} chars)")

Loaded 1 document(s) from CV.txt
 - CV.txt (3962 chars)


# Splitting document into Chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} document(s) into {len(chunks)} chunks")
print("\nExample chunk:\n" + "-" * 50)
print(chunks[0].page_content[:300], "...")

Split 1 document(s) into 12 chunks

Example chunk:
--------------------------------------------------
Name: Bushra Qadir
Title: Software Engineering Student & Aspiring ML Engineer
Location: Lahore, Pakistan
Email: s2024065009@umt.edu.pk
LinkedIn: linkedin.com/in/bushra-qadir ...


# Turn Chunks into Embeddings

In [6]:
#Checking available model
%pip install -q -U google-genai
print("Done. Restart kernel now.")

Note: you may need to restart the kernel to use updated packages.
Done. Restart kernel now.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Available models:")
for m in client.models.list():
    print(" -", m.name)

Available models:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.0-flash
 - models/gemini-2.0-flash-001
 - models/gemini-2.0-flash-lite-001
 - models/gemini-2.0-flash-lite
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-pro-preview
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.5-flash
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - models/gemini-3.1-flash-tts-preview
 

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
sample_vector = embeddings.embed_query("What is the highest education of Bushra?")
print("Each embedding is a vector of length:", len(sample_vector))
print("First 5 numbers:", [round(x, 4) for x in sample_vector[:5]])

Each embedding is a vector of length: 3072
First 5 numbers: [-0.0014, 0.0079, 0.0083, -0.0572, -0.0163]


# Storing chinks into vector databases

In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("vectorstore")
print(f"Indexed {vectorstore.index.ntotal} chunks and saved to ./vectorstore/")

Indexed 12 chunks and saved to ./vectorstore/


# Retrieve

In [10]:
from pathlib import Path

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

found = retriever.invoke("What is the highest education of Bushra?")
print(f"Retrieved {len(found)} chunks. Top match:\n" + "-"*50)
print("Source:", Path(found[0].metadata["source"]).name)
print(found[0].page_content[:300], "...")

Retrieved 3 chunks. Top match:
--------------------------------------------------
Source: CV.txt
Name: Bushra Qadir
Title: Software Engineering Student & Aspiring ML Engineer
Location: Lahore, Pakistan
Email: s2024065009@umt.edu.pk
LinkedIn: linkedin.com/in/bushra-qadir ...


# Debug View

In [11]:
question = "What ML projects has Bushra built?"
docs = retriever.invoke(question)
print(f"Retrieved {len(docs)} chunks for: {question!r}\n" + "="*60)
for i, d in enumerate(docs, 1):
    print(f"\n[{i}] source: {Path(d.metadata['source']).name}")
    print(d.page_content[:220].strip())

Retrieved 3 chunks for: 'What ML projects has Bushra built?'

[1] source: CV.txt
Name: Bushra Qadir
Title: Software Engineering Student & Aspiring ML Engineer
Location: Lahore, Pakistan
Email: s2024065009@umt.edu.pk
LinkedIn: linkedin.com/in/bushra-qadir

[2] source: CV.txt
KEY PROJECTS

1. Heart Disease Prediction System (ML)
   Built a machine learning classification model to predict heart disease risk
   from patient clinical data. Performed data preprocessing, feature selection,
   and

[3] source: CV.txt
2. Student Performance Prediction System (ML)
   Developed a predictive model analyzing academic and behavioral features to
   forecast student performance outcomes, aimed at enabling early academic
   intervention and s


# Writing grounded Prompt

In [15]:
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are CVBot, an assistant that answers questions about Bushra Qadir "
     "using ONLY the context below.\n"
     "- Use only facts found in the context; never use outside knowledge.\n"
     "- You MAY combine facts from several context chunks to give a complete answer.\n"
     "- If the answer is genuinely not in the context, say: "
     "\"I don't have that information in the CV.\"\n"
     "- Be concise and quote details (dates, degree, project names) exactly as written.\n\n"
     "Context:\n{context}"),
    ("human", "{input}"),
])
print("Prompt ready.")

Prompt ready.


# Building the RAG Chain

In [16]:
try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ModuleNotFoundError:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_chain = create_stuff_documents_chain(llm, answer_prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

result = rag_chain.invoke({"input": "What is the highest education of Bushra?"})
print(result["answer"])

Bushra's highest education listed is a BS in Software Engineering from the University of Management and Technology, Lahore, with an expected graduation in 2026.


# Test Refusal

In [17]:
result = rag_chain.invoke({"input": "Who is the president of Pakistan?"})
print(result["answer"])

I don't have that information in the CV.
